In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [2]:
class LSTMLayer(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # LSTM parameters
        self.W_u = nn.Linear(input_size + hidden_size, hidden_size)  # Input gate
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)  # Forget gate
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)  # Output gate
        self.W_c = nn.Linear(input_size + hidden_size, hidden_size)  # Cell candidate
        
        # Output layer
        self.output_layer = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, a_prev, c_prev):
        combined = torch.cat((x, a_prev), dim=1)    # [batch_size, input_size + hidden_size]
        
        c_tilde_t = torch.tanh(self.W_c(combined))  # [batch_size, hidden_size]
        u_t = torch.sigmoid(self.W_u(combined))     # [batch_size, hidden_size]
        f_t = torch.sigmoid(self.W_f(combined))     # [batch_size, hidden_size]
        o_t = torch.sigmoid(self.W_o(combined))     # [batch_size, hidden_size]
        
        c_t = f_t * c_prev + u_t * c_tilde_t        # [batch_size, hidden_size]
        a_t = o_t * torch.tanh(c_t)                 # [batch_size, hidden_size]
        y_t = self.output_layer(a_t)                # [batch_size, output_size]

        return a_t, c_t, y_t

class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size, pad_token_id, lr=0.001):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm_layer = LSTMLayer(embedding_dim, hidden_size, output_size)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token_id)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        self.pad_token_id = pad_token_id

    def forward(self, x):
        batch_size, seq_len = x.size()

        x_mask = (x == self.pad_token_id).unsqueeze(-1)                             # [batch_size, seq_len, 1]
        x_emb = self.embedding(x)                                                   # [batch_size, seq_len, embedding_dim]
        x_emb = x_emb.masked_fill(x_mask, 0)                                             

        a_prev = torch.zeros(batch_size, self.lstm_layer.hidden_size)
        c_prev = torch.zeros(batch_size, self.lstm_layer.hidden_size)

        outputs = []
        for t in range(seq_len):
            x_t = x_emb[:, t, :]                                                    # [batch_size, embedding_dim]
            a_prev, c_prev, y_t = self.lstm_layer(x_t, a_prev, c_prev)              # [batch_size, hidden_size], [batch_size, hidden_size], [batch_size, output_size]
            outputs.append(y_t)

        outputs = torch.stack(outputs, dim=1)  # [batch_size, seq_len, output_size]
        return outputs
    

    def backward(self, y_pred, y_true):
        loss = self.loss_fn(y_pred.view(-1, y_pred.size(-1)), y_true.view(-1))
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.item()
    
    def fit(self, x, y, epochs=10, print_every=1):
        for epoch in range(epochs):
            y_pred = self.forward(x)
            loss = self.backward(y_pred, y)
            if (epoch + 1) % print_every == 0:
                print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

src_texts = [
    "I love machine learning.",
    "Transformers are powerful models.",
    "I learn deep learning."
]

src_tokens = tokenizer(src_texts, padding=True, truncation=True, return_tensors="pt")

print("Source tokens:\n", src_tokens['input_ids'])
print("Source tokens shape:", src_tokens['input_ids'].shape)

Source tokens:
 tensor([[   40,  1842,  4572,  4673,    13, 50256],
        [41762,   364,   389,  3665,  4981,    13],
        [   40,  2193,  2769,  4673,    13, 50256]])
Source tokens shape: torch.Size([3, 6])


In [4]:
lstm = LSTM(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=64,
    hidden_size=128,
    output_size=tokenizer.vocab_size,
    pad_token_id=tokenizer.pad_token_id,
    lr=0.001
)
lstm.fit(src_tokens['input_ids'], src_tokens['input_ids'], epochs=100, print_every=10)

Epoch 10/100, Loss: 10.3525
Epoch 20/100, Loss: 9.2913
Epoch 30/100, Loss: 6.7886
Epoch 40/100, Loss: 3.7254
Epoch 50/100, Loss: 2.0079
Epoch 60/100, Loss: 1.2000
Epoch 70/100, Loss: 0.7196
Epoch 80/100, Loss: 0.4260
Epoch 90/100, Loss: 0.2732
Epoch 100/100, Loss: 0.1926


In [5]:
def generate(model, tokenizer, prompt, max_length=20):
    model.eval()
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    generated = input_ids

    with torch.no_grad():
        for _ in range(max_length):
            outputs = model(generated)
            next_token_logits = outputs[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            generated = torch.cat((generated, next_token_id), dim=1)

    return tokenizer.decode(generated[0], skip_special_tokens=True)

In [6]:
for text in src_texts:
    print(f"Prompt: {text}")
    print("Generated:", generate(lstm, tokenizer, text))
    print()

Prompt: I love machine learning.
Generated: I love machine learning.....................

Prompt: Transformers are powerful models.
Generated: Transformers are powerful models.....................

Prompt: I learn deep learning.
Generated: I learn deep learning.....................

